### 3. Práctica: Representaciones 
#### Matrices dispersas y búsqueda de documentos

Este apartado requiere que construyas un motor de búsqueda entre documentos comparando el rendimiento de una Bolsa de Palabras (BoW) y TF-IDF para procesar un documento "tramposo" (documento con muchas palabras que aportan poco significado o valor temático):

1. Define una lista de 5 documentos cortos divididos en dos temas contrastantes.
    - Ej: 3 de Revolución Rusa y 2 de comida vegana.
2. Crea una query "tramposa", esto es, crea un documento dirigido a alguna temática pero repitiendo intencionalmente palabras comunes o verbos genéricos que aparezcan en los documentos de la otra temática.
3. Vectoriza para crear una BoW y calcula la similitud coseno entre la query y los 5 documentos
4. Repite el proceso usando TF-IDF
5. Imprime un DataFrame o tabla comparativa que muestre los scores de similitud de BoW y TF-IDF del query contra los 5 documentos.
    - ¿Cambió el documento clasificado como "más similar/relevante" al pasar de BoW a TF-IDF? Identifica el cambio si lo hubo.
    - Explica brevemente, basándote en la penalización idf (Inverse Document Frequency), cómo y por qué TF-IDF procesó de manera distinta las palabras de tu "trampa léxica" en comparación con BoW.

In [25]:
!{sys.executable} -m pip install nltk

In [14]:
import sys
!{sys.executable} -m pip install scikit-learn

   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.0 MB ? eta -:--:--
   ------------------- -------------------- 3.9/8.0 MB 15.0 MB/s eta 0:00:01
   -------------------------------------- - 7.6/8.0 MB 16.1 MB/s eta 0:00:01
   ---------------------------------------- 8.0/8.0 MB 13.4 MB/s  0:00:00

   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   -------------------- ------------------- 1/2 [scikit-learn]
   

In [26]:
import nltk
nltk.download('punkt_tab')
import numpy as np
import pandas as pd

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\luisi\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


In [20]:
nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\luisi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [10]:
doc_1="El ramadán es el noveno mes del calendario islámico, respetado por musulmanes como el mes del ayuno, oración, reflexión y comunidad. Cada año el mes en el que se celebra el Ramadán cambia en torno al mes lunar. Es una conmemoración de la primera revelación de Mahoma. El cumplimiento anual del ramadán está considerado como uno de los cinco pilares del islam y su duración es de veintinueve a treinta días, a partir de la luna creciente hasta la próxima luna creciente."
doc_2="El Ramadán es el mes más sagrado para los seguidores del islam. Durante este periodo, quienes están en condiciones de hacerlo deben abstener se de comer, beber, fumar y mantener relaciones sexuales desde el amanecer hasta el atardecer. El propósito, de acuerdo con la tradición, es fortalecer la relación con Dios, practicar la autodisciplina y desarrollar empatía hacia quienes tienen menos recursos. De acuerdo con información de Middle East Eye, el ayuno es uno de los cinco pilares del islam, junto con La declaración de fe La oración La caridad La peregrinación a La Meca. Los musulmanes creen que durante este mes fueron revelados los primeros versículos del Corán al profeta Mahoma, hace más de 1.400 años."
doc_3="La física nuclear es una rama de la física que se ocupa del estudio de los núcleos atómicos, las partículas subatómicas y las interacciones nucleares. Se centra en comprender la estructura y propiedades de los núcleos atómicos, así como las fuerzas y reacciones nucleares que ocurren en ellos. La física nuclear abarca una amplia gama de temas, que incluyen la desintegración radioactiva, la fisión nuclear, la fusión nuclear, la radiactividad, las interacciones de partículas cargadas con la materia, las reacciones nucleares inducidas y la producción de energía a través de procesos nucleares. También se investiga la formación y desintegración de isótopos y la generación de elementos en el Universo, así como la radiación y sus efectos sobre la materia y los seres vivos. Los avances en la física nuclear han llevado al desarrollo de aplicaciones prácticas en diversos campos, como la generación de energía nuclear, la medicina nuclear, la datación por radiocarbono, la investigación en astrofísica y la producción de materiales y radioisótopos para uso industrial y médico."
doc_4="La física nuclear es una rama de la física que estudia las propiedades, comportamiento e interacciones de los núcleos atómicos. En un contexto más amplio, se define la física nuclear y de partículas como la rama de la física que estudia la estructura fundamental de la materia y las interacciones entre las partículas subatómicas. La física nuclear es conocida mayoritariamente por el aprovechamiento de la energía nuclear en centrales nucleares y en el desarrollo de armas nucleares, tanto de fisión nuclear como de fusión nuclear, pero este campo ha dado lugar a aplicaciones en diversos campos, incluyendo medicina nuclear e imágenes por resonancia magnética, ingeniería de implantación de iones en materiales y datación por radiocarbono en geología y arqueología."
doc_5="La física nuclear es una rama fundamental de la física que estudia el núcleo atómico, sus componentes y las interacciones que en él ocurren. A lo largo de más de un siglo, esta disciplina ha evolucionado de simples teorías y descubrimientos experimentales a una ciencia aplicada que influye en múltiples aspectos de la vida moderna. En este artículo exploraremos en detalle qué es la física nuclear, sus fundamentos, su desarrollo histórico y las aplicaciones que han transformado sectores tan diversos como la medicina, la energía, la industria y la investigación científica. La física nuclear se encarga de estudiar las propiedades y la estructura de los núcleos de los átomos. Los núcleos están compuestos por protones y neutrones, partículas subatómicas que interactúan mediante la fuerza nuclear fuerte. Este campo se originó a comienzos del siglo XX, cuando experimentos pioneros revelaron que la mayor parte de la masa de un átomo se concentraba en su núcleo y que las interacciones internas eran mucho más complejas de lo que se pensaba en el modelo atómico clásico. La comprensión de estas interacciones abrió la puerta a aplicaciones revolucionarias que hoy forman parte de nuestro día a día. La importancia de la física nuclear radica en la capacidad de explicar fenómenos tanto a nivel microscópico como macroscópico. Desde la estabilidad de la materia hasta los procesos energéticos que ocurren en el sol, la física nuclear ofrece respuestas a preguntas fundamentales sobre la estructura de la materia y las leyes que rigen el universo."

In [11]:
documents = [doc_1, doc_2, doc_3, doc_4, doc_5]

In [27]:
import re
from nltk.tokenize import word_tokenize


def simple_preprocess(text: str):
    tokens = word_tokenize(text.lower(), language="spanish")
    # Ignoramos signos de puntuación y palabras de longitud 1
    return [word for word in tokens if word.isalnum() and len(word) > 1 and not re.match(r"\d+", word)]

In [28]:
from sklearn.feature_extraction.text import CountVectorizer

In [29]:
vectorizer = CountVectorizer(tokenizer=simple_preprocess, token_pattern=None)

In [31]:
bag_of_words_corpus = vectorizer.fit_transform(documents)

In [32]:
diccionario = vectorizer.vocabulary_

In [47]:
bag_of_words_corpus.toarray()

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [1, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 1, 1, 1]], shape=(5, 292))

In [36]:
print(len(bag_of_words_corpus.toarray()))
len(bag_of_words_corpus.toarray()[1])

5


292

In [37]:
def create_bow_dataframe(docs_raw: list, titles: list[str], vectorizer) -> pd.DataFrame:
    # fit_transform ajusta el vocabulario y crea la matriz en un solo paso
    matrix = vectorizer.fit_transform(docs_raw)

    # Podemos crear el DataFrame directamente pasando la matriz a un array tradicional
    # vectorizer.get_feature_names_out() nos da la lista de palabras en el orden exacto de las columnas
    df = pd.DataFrame(
        matrix.toarray(), index=titles, columns=vectorizer.get_feature_names_out()
    )
    return df

In [41]:
titles = ["RAMADAN(WIKIPEDIA)", "RAMADAN(EL IMPARCIAL)", "FISICA NUCLEAR(ENERGIA NUCLEAR)","FISICA NUCLEAR(ESTUDYANDO)","FISICA NUCLEAR(WIKIPEDIA)"]
docs_matrix = create_bow_dataframe(
    documents,
    titles,
    vectorizer=CountVectorizer(tokenizer=simple_preprocess, token_pattern=None)#, binary=True),
)

In [42]:
docs_matrix

,abarca,abrió,abstener,acuerdo,al,amanecer,amplia,amplio,anual,aplicaciones,...,uno,uso,veintinueve,versículos,vida,vivos,xx,átomo,átomos,él
RAMADAN(WIKIPEDIA),0,0,0,0,1,0,0,0,1,0,...,1,0,1,0,0,0,0,0,0,0
RAMADAN(EL IMPARCIAL),0,0,1,2,1,1,0,0,0,0,...,1,0,0,1,0,0,0,0,0,0
FISICA NUCLEAR(ENERGIA NUCLEAR),1,0,0,0,1,0,1,0,0,1,...,0,1,0,0,0,1,0,0,0,0
FISICA NUCLEAR(ESTUDYANDO),0,0,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
FISICA NUCLEAR(WIKIPEDIA),0,1,0,0,0,0,0,0,0,2,...,0,0,0,0,1,0,1,1,1,1


In [ ]:
documents = [doc_1, doc_2, doc_3, doc_4, doc_5]

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [45]:
docs_matrix_tfidf = create_bow_dataframe(
    documents, titles, TfidfVectorizer(tokenizer=simple_preprocess, token_pattern=None)
)

In [46]:
docs_matrix_tfidf

,abarca,abrió,abstener,acuerdo,al,amanecer,amplia,amplio,anual,aplicaciones,...,uno,uso,veintinueve,versículos,vida,vivos,xx,átomo,átomos,él
RAMADAN(WIKIPEDIA),0.000000,0.000000,0.000000,0.000000,0.072432,0.000000,0.000000,0.00000,0.108154,0.000000,...,0.087258,0.000000,0.108154,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
RAMADAN(EL IMPARCIAL),0.000000,0.000000,0.086803,0.173606,0.058133,0.086803,0.000000,0.00000,0.000000,0.000000,...,0.070032,0.000000,0.000000,0.086803,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
FISICA NUCLEAR(ENERGIA NUCLEAR),0.060436,0.000000,0.000000,0.000000,0.040475,0.000000,0.060436,0.00000,0.000000,0.040475,...,0.000000,0.060436,0.000000,0.000000,0.000000,0.060436,0.000000,0.000000,0.000000,0.000000
FISICA NUCLEAR(ESTUDYANDO),0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.08314,0.000000,0.055680,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
FISICA NUCLEAR(WIKIPEDIA),0.000000,0.048321,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.064722,...,0.000000,0.000000,0.000000,0.000000,0.048321,0.000000,0.048321,0.048321,0.048321,0.048321


In [49]:
#Ahora crearemos el documento "tramposo" que contenga algunas palabras similares a las que esten relacionados con ambos documentos tanto del Ramadán
#como de la fisica nuclear
tricky_query = "El Ramadán es una festividad musulmana que se centra en la conmemoración de la primera revelación de mahoma, a través de la interacción entre la comunidad, la disciplina abarca tradiciones como el ayuno y la oración. Esta festividad influye en la manera que los creyentes observan y se sienten respecto a las recompensas espirituales, siendo que es una celebridad donde los creyentes se centran en la fé y la primera revelación de Mahoma. Esta ocurre en el noveno mes del calendario islámico, donde se transforma su vide e influyen en la percepci+on de la misma."

In [63]:
updated_docs = documents.copy()
updated_docs.append(tricky_query)
updated_titles = titles + ["RAMADAN TRICKY"]

In [64]:
updated_matrix = create_bow_dataframe(
    updated_docs,
    updated_titles,
    vectorizer=TfidfVectorizer(tokenizer=simple_preprocess, token_pattern=None),
)

In [66]:
from sklearn.metrics.pairwise import cosine_similarity

In [67]:
for doc_title in updated_titles:
    current_doc_values = updated_matrix.loc[doc_title].values.reshape(1, -1)
    # Seleccionamos [0][0] para extraer el valor numérico (float) de la matriz de resultado
    sim = cosine_similarity(current_doc_values, doc_tricky_values)[0][0]
    print(f"Similitud entre RAMADANTRICKY/{doc_title} = {sim:.2f}")

Similitud entre RAMADANTRICKY/RAMADAN(WIKIPEDIA) = 0.41
Similitud entre RAMADANTRICKY/RAMADAN(EL IMPARCIAL) = 0.35
Similitud entre RAMADANTRICKY/FISICA NUCLEAR(ENERGIA NUCLEAR) = 0.48
Similitud entre RAMADANTRICKY/FISICA NUCLEAR(ESTUDYANDO) = 0.38
Similitud entre RAMADANTRICKY/FISICA NUCLEAR(WIKIPEDIA) = 0.50
Similitud entre RAMADANTRICKY/RAMADAN TRICKY = 1.00


In [70]:
docs_matrixbownew = create_bow_dataframe(
    updated_docs,
    updated_titles,
    vectorizer=CountVectorizer(tokenizer=simple_preprocess, token_pattern=None)#, binary=True),
)

In [73]:
for doc_title in updated_titles:
    bow_docs_values = docs_matrixbownew.loc[doc_title].values.reshape(1, -1)
    # Seleccionamos [0][0] para extraer el valor numérico (float) de la matriz de resultado
    sim = cosine_similarity(bow_docs_values, doc_tricky_values)[0][0]
    print(f"Similitud entre SAMPLE/{doc_title} = {sim:.2f}")

Similitud entre SAMPLE/RAMADAN(WIKIPEDIA) = 0.49
Similitud entre SAMPLE/RAMADAN(EL IMPARCIAL) = 0.49
Similitud entre SAMPLE/FISICA NUCLEAR(ENERGIA NUCLEAR) = 0.58
Similitud entre SAMPLE/FISICA NUCLEAR(ESTUDYANDO) = 0.49
Similitud entre SAMPLE/FISICA NUCLEAR(WIKIPEDIA) = 0.60
Similitud entre SAMPLE/RAMADAN TRICKY = 0.94


### 2.Busqueda de sesgos
##### 1.-Descarga el modelo glove-wiki-gigaword-100 con la api de gensim y ejecuta el siguiente código:
print(word_vectors.most_similar(positive=['man', 'profession'], negative=['woman']))

print()

print(word_vectors.most_similar(positive=['woman', 'profession'], negative=['man']))
##### 2.-Identifica las diferencias en la lista de palabras asociadas a hombres/mujeres y profesiones, explica como esto reflejaría un sesgo de genero.
##### 3.-Utiliza la función .most_similar() para identificar analogías que exhiba algún tipo de sesgo de los vectores pre-entrenados.
Explica brevemente que sesgo identificar
Si fuera tu trabajo crear un modelo ¿Como mitigarías estos sesgos al crear vectores de palabras?

In [74]:
!pip install gensim

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
    --------------------------------------- 0.5/24.4 MB 1.7 MB/s eta 0:00:15
   - -------------------------------------- 0.8/24.4 MB 1.9 MB/s eta 0:00:13
   -- ------------------------------------- 1.3/24.4 MB 2.0 MB/s eta 0:00:12
   --- ------------------------------------ 1.8/24.4 MB 2.1 MB/s eta 0:00:11
   --- ------------------------------------ 2.4/24.4 MB 2.2 MB/s eta 0:00:11
   ---- ----------------------------------- 2.9/24.4 MB 2.2 MB/s eta 0:00:10
   ----- ---------------------------------- 3.4/24.4 MB 2.3 MB/s eta 0:00:10
   ------ --------------------------------- 3.9/24.4 MB 2.3 MB/s eta 0:00:09
   ------- -------------------------------- 4.5/24.4 MB 2.3 MB/s eta 0:00:09
   -------- ---------------

In [75]:
import gensim.downloader as gensim_api

In [76]:
gensim_api.info(name_only=True)

{'corpora': ['semeval-2016-2017-task3-subtaskBC',
  'semeval-2016-2017-task3-subtaskA-unannotated',
  'patent-2017',
  'quora-duplicate-questions',
  'wiki-english-20171001',
  'text8',
  'fake-news',
  '20-newsgroups',
  '__testing_matrix-synopsis',
  '__testing_multipart-matrix-synopsis'],
 'models': ['fasttext-wiki-news-subwords-300',
  'conceptnet-numberbatch-17-06-300',
  'word2vec-ruscorpora-300',
  'word2vec-google-news-300',
  'glove-wiki-gigaword-50',
  'glove-wiki-gigaword-100',
  'glove-wiki-gigaword-200',
  'glove-wiki-gigaword-300',
  'glove-twitter-25',
  'glove-twitter-50',
  'glove-twitter-100',
  'glove-twitter-200',
  '__testing_word2vec-matrix-synopsis']}

In [77]:
word_vectors = gensim_api.load("glove-wiki-gigaword-100")

[==================================================] 100.0% 128.1/128.1MB downloaded


In [78]:
print(word_vectors.most_similar(positive=['man', 'profession'], negative=['woman']))
print()
print(word_vectors.most_similar(positive=['woman', 'profession'], negative=['man']))

[('practice', 0.6156836152076721), ('knowledge', 0.6129590272903442), ('teaching', 0.5949095487594604), ('skill', 0.5886170864105225), ('reputation', 0.588079571723938), ('philosophy', 0.5868663191795349), ('work', 0.5848589539527893), ('skills', 0.5772278904914856), ('discipline', 0.576593816280365), ('mind', 0.5739315152168274)]

[('professions', 0.6473134756088257), ('practitioner', 0.5966603755950928), ('nursing', 0.5942842364311218), ('vocation', 0.5698666572570801), ('teaching', 0.5623518824577332), ('childbirth', 0.543552815914154), ('academic', 0.5408717393875122), ('teacher', 0.5401058197021484), ('educator', 0.5207306742668152), ('qualifications', 0.5143449902534485)]


### ANÁLISIS

Si analizamos ambos conjuntos de palabras podemos observar una gran diferencia, pues las profesiones que se exponen en ambas si que son bastantes distintas entre sí, como la de enfermera o filosofía en el caso de los hombres, considero que el sesgo se observa más en las palabras relacionadas, pues se le relaciona más a la mujer en profesiones parto, cosa que personalmente me parecen rara además de mencionar vocación y cualificaciones, mientras que para los hombres resaltan la disciplina el conocimiento la habilidad y la relación con la reputación, el hecho de incluir estas características solo en la parte para hombre.

Esto para mi indica que mientras que para los hombres las cualidades indican reconocimiento y disciplina para las mujeres se enfocan más en su lado de cuidados e indicando que estas no sean acordes a dichas características sobretodo me sorprende como no aparece la habilidad en ambos y como en mujeres aparecen como doble palabra respecto a dar orientación/ eduación y para los hombres aparezca habilidad y habilidades en general.

In [79]:
print(word_vectors.most_similar(positive=['young', 'profession'], negative=['old']))
print()
print(word_vectors.most_similar(positive=['old', 'profession'], negative=['young']))

[('professionals', 0.6508859395980835), ('professions', 0.6299289464950562), ('practitioners', 0.5842034816741943), ('physicians', 0.5603626370429993), ('peers', 0.5564031600952148), ('teaching', 0.5548090934753418), ('academics', 0.5529879331588745), ('society', 0.5475229024887085), ('careers', 0.5440095663070679), ('creative', 0.5376880168914795)]

[('practice', 0.5177038311958313), ('tradition', 0.5106244087219238), ('50-year', 0.4880888760089874), ('70-year', 0.48156213760375977), ('history', 0.4775371253490448), ('institution', 0.47672349214553833), ('40-year', 0.47626668214797974), ('accountant', 0.474350243806839), ('apprenticeship', 0.47292569279670715), ('nursing', 0.46711012721061707)]


### ANÁLISIS
En esta ocasión consulte una página para revisitar cuales han sido y siguen siendo los prejuicios mas comúnes en el caso anterior vimos un sesgo sexista, en este decidí optar por si este modelo contiene sesgos relacionados al edadismo, siento que no es tan común notarlo, sin embargo considero que es uno de los más fuertes en la sociedad actual y revisaremos que nos arrojó el modelo.

En este caso seguimos con la relación a las profesiones pues siempre se considera que a cierta edad si eres joven o anciano te corresponden ciertos trabajos y se te limita a otros por lo cual veremos si el modelo tiene dicho sesgo.

De inicio notamos que para la sección de profesiones se da nuevamente enfermera y se quita la característica de creativo así como de agregar a contadores junto a la caracteristica de viejo y se quita la de enseñar en esta misma manteniendola para la de joven además de indicar médicos solo para los jóvenes.

Apesar de ello me agrada como la caracteristica de profesión o profesionistas se mantiene en ambas.


### COMO EVITAR SESGOS

Yo considero que si tuviera que hacer algun modelo no importa que haga se mantendrán sesgos pues considero que un entrenamiento es mas lento que la actualización en el lenguaje e ideas de la sociedad en general, podríamos intentarlo enviando textos nuevos y certificados respecto a una actualización en la toma de relaciones semánticas, enseñandole nuevos documentos parecidos o cuyas frases ocntengan estas nuevas palabras e ideas intentando corregir el modelo a través del entrenamiento y nutrición con textos de una forma constante y supervisada.

Aunque vuelvo a lo mismo considero que por la cantidad de texto y de revisiones como contenido nuevo que no es comúnmente usado se mantendrá una leve capa o leve tiempo donde los sesgos o prejuicios se mantengan sea largo o corto se mantendrá por el simple hehco de tiempo de entrenamiento y análisis del modelo para generar dicha semántica,

A pesar de ello creo que una buena medida es esa, ir explorando de manera supervisaqda estos textos nuevos que van conteniendo nuevo vocabulario como ideas o conjunciones de frases que uiza en otros anteiores no se vean.

### REFERENCIAS
colaboradores de Wikipedia. (2026, March 29). Física nuclear. Wikipedia, La Enciclopedia Libre. https://es.wikipedia.org/wiki/F%C3%ADsica_nuclear

Cruzito. (2025, March 8). ¿Qué es la Física Nuclear y cuáles son sus Aplicaciones? | Estudyando. Estudyando. https://estudyando.com/que-es-la-fisica-nuclear-y-cuales-son-sus-aplicaciones/

La física nuclear: una introducción simple. (n.d.). [Video]. https://energia-nuclear.net/fisica/fisica-nuclear#google_vignette

colaboradores de Wikipedia. (2026, March 19). Ramadán. Wikipedia, La Enciclopedia Libre. https://es.wikipedia.org/wiki/Ramad%C3%A1n

Zarate, A. (2026, February 18). ¿Qué es el Ramadán, cómo se práctica y cuándo inicia? Todos los detalles sobre el mes más sagrado para los seguidores del islam. ¿Qué Es El Ramadán, Cómo Se Práctica Y Cuándo Inicia? Todos Los Detalles Sobre El Mes Más Sagrado Para Los Seguidores Del Islam. https://www.elimparcial.com/estilos/2026/02/18/que-es-el-ramadan-como-se-practica-y-cuando-inicia-todos-los-detalles-sobre-el-mes-mas-sagrado-para-los-seguidores-del-islam/

PSICOBLOG. (2026, March 19). Prejuicios SOCIALES: Ejemplos y CONSECUENCIAS Impactantes. https://psicoblog.org/prejuicios-sociales-ejemplos/#Tipos_Comunes_de_Prejuicios 